In [1]:
import pandas as pd
import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from mlflow.models import infer_signature
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# 데이터 로딩 및 전처리
data = pd.read_csv('../data/churn.csv')

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [3]:
X = data.drop(['Exited', 'RowNumber', 'CustomerId', 'Surname'], axis=1)
y = data['Exited']

In [4]:
categorical_features = ['Geography', 'Gender']
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

In [6]:
X_processed = preprocessor.fit_transform(X)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [8]:
# 하이퍼파라미터 탐색 공간 정의
space = {
    'max_depth': hp.choice('max_depth', range(3, 10)),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'n_estimators': hp.choice('n_estimators', range(50, 200)),
    'gamma': hp.uniform('gamma', 0, 5)
}

In [9]:
# objective 함수 정의
def objective(params):
    with mlflow.start_run(nested=True):
        model = xgb.XGBClassifier(
            use_label_encoder=False,
            eval_metric='logloss',
            **params
        )
        model.fit(X_train, y_train)
        
        # 예측값과 확률 분리
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        precision = precision_score(y_test, preds)
        recall = recall_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        roc_auc = roc_auc_score(y_test, probs)

        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'roc_auc': roc_auc
        })

        signature = infer_signature(X_train, probs)

        return {
            'loss': -roc_auc,
            'status': STATUS_OK,
            'model': model,
            'signature': signature
        }

In [10]:
# 실험 설정 및 fmin 수행
mlflow.set_experiment("practice5")

with mlflow.start_run(run_name="XGBoost") as run:
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=20,
        trials=trials
    )

    # best run 결과 추출
    best_result = sorted(trials.results, key=lambda x: x["loss"])[0]
    best_model = best_result["model"]
    signature = best_result["signature"]

    mlflow.log_params(best)
    mlflow.log_metric("best_roc_auc", -best_result["loss"])

    # 최종 best 모델 저장 (루트 run에)
    mlflow.xgboost.log_model(best_model, "best_xgboost_model", signature=signature)

    print(f"Best parameters: {best}")
    print(f"Best ROC-AUC: {-best_result['loss']:.4f}")

2026/04/22 12:02:58 INFO mlflow.tracking.fluent: Experiment with name 'practice5' does not exist. Creating a new experiment.


🏃 View run handsome-whale-506 at: http://127.0.0.1:5000/#/experiments/11/runs/e64389849ef344f5b42a872cb78d8a98

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/11

  5%|▌         | 1/20 [00:00<00:05,  3.34trial/s, best loss: -0.8671271203750766]

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:02:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:02:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🏃 View run dapper-turtle-526 at: http://127.0.0.1:5000/#/experiments/11/runs/92f128540e5343e49ee98b2a08b86455

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/11                    

🏃 View run honorable-shrew-934 at: http://127.0.0.1:5000/#/experiments/11/runs/9ddde4a1103648f69dc627c9e0d0aaff

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/11                    

 15%|█▌        | 3/20 [00:00<00:03,  5.05trial/s, best loss: -0.8717530334050614]

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:02:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:02:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



...

🏃 View run fearless-penguin-572 at: http://127.0.0.1:5000/#/experiments/11/runs/08fc3ae154164f46a9f2e183f6282881

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/11                     

🏃 View run mysterious-cod-704 at: http://127.0.0.1:5000/#/experiments/11/runs/9d18dd5cc1344d789075e79f3063b77d

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/11                     

 90%|█████████ | 18/20 [00:03<00:00,  5.60trial/s, best loss: -0.8756141625933613]

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:03:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:03:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🏃 View run skillful-shoat-567 at: http://127.0.0.1:5000/#/experiments/11/runs/091ad0aa08f14f53bebc305243e41a06

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/11                     

🏃 View run stately-toad-656 at: http://127.0.0.1:5000/#/experiments/11/runs/aa57c40fb80b4d5a80de2f22a446c9d4

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/11                     

100%|██████████| 20/20 [00:03<00:00,  5.59trial/s, best loss: -0.8756141625933613]

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:03:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

c:\Users\SSAFY\Desktop\TIL\09_data_analysis\g_mlflow\mlflow_env\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [12:03:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)



Best parameters: {'gamma': 2.6529225626746107, 'learning_rate': 0.10593584646990459, 'max_depth': 3, 'n_estimators': 9}
Best ROC-AUC: 0.8756
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/11/runs/3d7ade2c6a934795a188f69c0355048d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/11
